# fee

> Normalized block fee data for fast, reproducible visualization work.

In [ ]:
#| default_exp fee

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import gzip
import json
import math
from dataclasses import dataclass
from decimal import Decimal
from pathlib import Path
from typing import Any, Mapping, Sequence

SATS_PER_BTC = Decimal("100000000")
DEFAULT_PERCENTILES = (10, 25, 50, 75, 90, 99)


@dataclass(frozen=True)
class FeeTx:
    """A compact non-coinbase transaction record for fee analysis."""

    vsize_vb: int
    fee_sats: int

    @property
    def fee_sat_vb(self) -> float:
        if self.vsize_vb <= 0:
            return 0.0
        return self.fee_sats / self.vsize_vb


@dataclass(frozen=True)
class FeeBlock:
    """A compact, visualization-ready view of one block's fee market."""

    txs: tuple[FeeTx, ...]
    height: int | None = None
    block_hash: str | None = None
    block_time: int | None = None
    tx_count: int | None = None
    source: str | None = None

    @classmethod
    def from_rpc_block(cls, block: Mapping[str, Any], source: str | None = "bitcoin-core") -> "FeeBlock":
        """Build a compact fee block from a Bitcoin Core `getblock` verbosity=3 result."""
        fee_txs: list[FeeTx] = []
        for tx in block.get("tx", [])[1:]:
            vsize = int(tx.get("vsize") or 0)
            if vsize <= 0:
                continue

            fee_sats = _rpc_tx_fee_sats(tx)
            if fee_sats is None or fee_sats < 0:
                continue
            fee_txs.append(FeeTx(vsize_vb=vsize, fee_sats=fee_sats))

        return cls(
            txs=tuple(fee_txs),
            height=block.get("height"),
            block_hash=block.get("hash"),
            block_time=block.get("time"),
            tx_count=len(block.get("tx", [])) or None,
            source=source,
        )

    @classmethod
    def from_record(cls, record: Mapping[str, Any]) -> "FeeBlock":
        """Load a `FeeBlock` from the stable compact fixture schema."""
        txs = []
        for row in record.get("txs", []):
            if isinstance(row, Mapping):
                txs.append(FeeTx(vsize_vb=int(row["vsize_vb"]), fee_sats=int(row["fee_sats"])))
            else:
                txs.append(FeeTx(vsize_vb=int(row[0]), fee_sats=int(row[1])))

        return cls(
            txs=tuple(txs),
            height=record.get("height"),
            block_hash=record.get("block_hash") or record.get("hash"),
            block_time=record.get("block_time") or record.get("time"),
            tx_count=record.get("tx_count"),
            source=record.get("source"),
        )

    def to_record(self) -> dict[str, Any]:
        """Return a compact, deterministic JSON-serializable fixture record."""
        return {
            "schema": "slashbtc.fee_block.v1",
            "height": self.height,
            "block_hash": self.block_hash,
            "block_time": self.block_time,
            "tx_count": self.tx_count,
            "source": self.source,
            "txs": [[tx.vsize_vb, tx.fee_sats] for tx in self.txs],
        }


def fee_block_from_chain(chain: Any, height_or_hash: int | str) -> FeeBlock:
    """Fetch one block through `Chain` and return the compact fee view."""
    return FeeBlock.from_rpc_block(chain.block(height_or_hash, verbosity=3))


def load_fee_block(path: str | Path) -> FeeBlock:
    """Load a compact fixture from `.json` or `.json.gz`."""
    path = Path(path)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        return FeeBlock.from_record(json.load(f))


def save_fee_block(block: FeeBlock, path: str | Path) -> Path:
    """Save a compact fixture to `.json` or `.json.gz` with deterministic formatting."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "wt", encoding="utf-8") as f:
        json.dump(block.to_record(), f, sort_keys=True, separators=(",", ":"))
        f.write("\n")
    return path


def capture_fee_block(chain: Any, height_or_hash: int | str, path: str | Path | None = None) -> FeeBlock:
    """Fetch a real block once, optionally saving the compact fixture for future offline use."""
    block = fee_block_from_chain(chain, height_or_hash)
    if path is not None:
        save_fee_block(block, path)
    return block


def fee_rates(block: FeeBlock) -> list[float]:
    """Return non-coinbase fee rates in sat/vB."""
    return [tx.fee_sat_vb for tx in block.txs if tx.vsize_vb > 0 and tx.fee_sats >= 0]


def fee_percentiles(
    block: FeeBlock,
    percentiles: Sequence[int] = DEFAULT_PERCENTILES,
    weight_by: str = "tx",
) -> dict[int, float | None]:
    """Return fee-rate percentiles, weighted by transaction count or virtual size."""
    if weight_by not in {"tx", "vsize"}:
        raise ValueError("weight_by must be 'tx' or 'vsize'")
    if not block.txs:
        return {p: None for p in percentiles}

    ordered = sorted(block.txs, key=lambda tx: tx.fee_sat_vb)
    if weight_by == "tx":
        return {p: _nearest_rank([tx.fee_sat_vb for tx in ordered], p) for p in percentiles}

    total_vsize = sum(tx.vsize_vb for tx in ordered)
    out: dict[int, float | None] = {}
    for p in percentiles:
        threshold = total_vsize * p / 100
        seen = 0
        value = ordered[-1].fee_sat_vb
        for tx in ordered:
            seen += tx.vsize_vb
            if seen >= threshold:
                value = tx.fee_sat_vb
                break
        out[p] = value
    return out


def fee_histogram(
    block: FeeBlock,
    max_sat_vb: int = 200,
    bucket_width: int = 1,
    include_empty: bool = False,
) -> list[dict[str, Any]]:
    """Bucket transactions by floored sat/vB for charting.

    The overflow bucket has `bucket_lower_sat_vb == max_sat_vb` and
    `bucket_upper_sat_vb is None`, which represents `max_sat_vb+`.
    """
    if max_sat_vb <= 0:
        raise ValueError("max_sat_vb must be positive")
    if bucket_width <= 0:
        raise ValueError("bucket_width must be positive")

    buckets: dict[int, dict[str, Any]] = {}
    if include_empty:
        for lower in range(0, max_sat_vb, bucket_width):
            buckets[lower] = _empty_bucket(lower, lower + bucket_width)
        buckets[max_sat_vb] = _empty_bucket(max_sat_vb, None)

    for tx in block.txs:
        lower = int(math.floor(tx.fee_sat_vb / bucket_width) * bucket_width)
        if lower >= max_sat_vb:
            lower = max_sat_vb
            upper = None
        else:
            upper = lower + bucket_width
        bucket = buckets.setdefault(lower, _empty_bucket(lower, upper))
        bucket["tx_count"] += 1
        bucket["total_vsize_vb"] += tx.vsize_vb
        bucket["total_fees_sats"] += tx.fee_sats

    rows = []
    for lower in sorted(buckets):
        bucket = buckets[lower]
        if bucket["tx_count"]:
            bucket["mean_fee_sat_vb"] = bucket["total_fees_sats"] / bucket["total_vsize_vb"]
        if include_empty or bucket["tx_count"]:
            rows.append(bucket)
    return rows


def equal_vsize_fee_buckets(block: FeeBlock, n_bins: int = 24) -> tuple[list[float], list[int]]:
    """Bucket txs into `n_bins` equal-vsize chunks ordered low to high feerate."""
    rows = sorted(((tx.vsize_vb, tx.fee_sat_vb) for tx in block.txs), key=lambda r: r[1])
    if not rows:
        return [], []

    total = sum(vsize for vsize, _ in rows)
    target = total / n_bins
    fee_rates_out: list[float] = []
    vsizes_out: list[int] = []
    cur_vsize = 0
    cur_weighted_fee = 0.0

    for vsize, rate in rows:
        cur_vsize += vsize
        cur_weighted_fee += rate * vsize
        if cur_vsize >= target:
            fee_rates_out.append(cur_weighted_fee / cur_vsize)
            vsizes_out.append(cur_vsize)
            cur_vsize = 0
            cur_weighted_fee = 0.0

    if cur_vsize:
        fee_rates_out.append(cur_weighted_fee / cur_vsize)
        vsizes_out.append(cur_vsize)
    return fee_rates_out, vsizes_out


def fee_summary(block: FeeBlock, weight_by: str = "tx") -> dict[str, Any]:
    """Return the headline numbers needed by block fee distribution panels."""
    rates = fee_rates(block)
    percentiles = fee_percentiles(block, weight_by=weight_by)
    positive_rates = [rate for rate in rates if rate > 0]
    tail = overpayment_tail(block)

    return {
        "height": block.height,
        "block_hash": block.block_hash,
        "block_time": block.block_time,
        "tx_count": block.tx_count or len(block.txs) + 1,
        "non_coinbase_tx_count": len(block.txs),
        "total_vsize_vb": sum(tx.vsize_vb for tx in block.txs),
        "total_fees_sats": sum(tx.fee_sats for tx in block.txs),
        "min_fee_sat_vb": min(rates) if rates else None,
        "max_fee_sat_vb": max(rates) if rates else None,
        "clearing_fee_sat_vb": min(positive_rates) if positive_rates else (min(rates) if rates else None),
        "percentiles": percentiles,
        "overpayment_tail": tail,
    }


def overpayment_tail(block: FeeBlock, share: float = 0.05) -> dict[str, float | int | None]:
    """Return count and threshold for the top fee-rate tail."""
    if not 0 < share < 1:
        raise ValueError("share must be between 0 and 1")
    rates = fee_rates(block)
    positive_rates = [rate for rate in rates if rate > 0]
    if not positive_rates:
        return {"threshold_fee_sat_vb": None, "tx_count": 0, "share": 0.0}

    threshold = _nearest_rank(sorted(positive_rates), 100 * (1 - share))
    count = sum(1 for rate in positive_rates if rate >= threshold)
    return {
        "threshold_fee_sat_vb": threshold,
        "tx_count": count,
        "share": count / len(rates),
    }


def _rpc_tx_fee_sats(tx: Mapping[str, Any]) -> int | None:
    if "fee" in tx:
        return _btc_to_sats(tx["fee"])

    try:
        input_sats = sum(_btc_to_sats(vin["prevout"]["value"]) for vin in tx.get("vin", []))
        output_sats = sum(_btc_to_sats(vout["value"]) for vout in tx.get("vout", []))
    except KeyError:
        return None
    return input_sats - output_sats


def _btc_to_sats(value: Any) -> int:
    return int((Decimal(str(value)) * SATS_PER_BTC).to_integral_exact())


def _empty_bucket(lower: int, upper: int | None) -> dict[str, Any]:
    return {
        "bucket_lower_sat_vb": lower,
        "bucket_upper_sat_vb": upper,
        "tx_count": 0,
        "total_vsize_vb": 0,
        "total_fees_sats": 0,
        "mean_fee_sat_vb": None,
    }


def _nearest_rank(values: Sequence[float], percentile: float) -> float | None:
    if not values:
        return None
    if percentile <= 0:
        return values[0]
    if percentile >= 100:
        return values[-1]
    index = math.ceil(percentile / 100 * len(values)) - 1
    return values[max(0, min(index, len(values) - 1))]


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()